# Domain C: Recommendation - Exploration

This notebook explores the 10 approaches for recommendation systems.

## Setup

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.domain_c_recommendation.data_generator import create_recommendation_dataset
from src.domain_c_recommendation.approach_01_popularity import PopularityRecommender
from src.domain_c_recommendation.approach_02_collaborative_filtering import UserBasedCF, ItemBasedCF
from src.domain_c_recommendation.approach_04_matrix_factorization import SVDRecommender
from src.core.metrics import compute_ranking_metrics

## Generate Dataset

In [ ]:
dataset = create_recommendation_dataset(n_users=500, n_items=200)

print(f"Users: {dataset['n_users']}")
print(f"Items: {dataset['n_items']}")
print(f"Train interactions: {len(dataset['train_interactions'])}")
print(f"Test users with relevant items: {len(dataset['test_items'])}")
print(f"Sparsity: {1 - len(dataset['train_interactions']) / (dataset['n_users'] * dataset['n_items']):.4f}")

## Visualize Rating Distribution

In [ ]:
ratings = [r for _, _, r in dataset['train_interactions']]

plt.figure(figsize=(8, 5))
plt.hist(ratings, bins=20, edgecolor='black')
plt.xlabel('Rating')
plt.ylabel('Count')
plt.title('Rating Distribution')
plt.show()

## Compare Approaches

In [ ]:
results = []
test_users = list(dataset['test_items'].keys())
y_true = [dataset['test_items'][u] for u in test_users]

# Popularity
pop_model = PopularityRecommender()
pop_model.train(dataset['train_matrix'], dataset['train_interactions'])
pop_pred = pop_model.predict(test_users, k=10)
pop_metrics = compute_ranking_metrics(y_true, pop_pred, k_values=[5, 10])
results.append({'Approach': 'Popularity', **pop_metrics})

# User-Based CF
ubcf_model = UserBasedCF({'k_neighbors': 30})
ubcf_model.train(dataset['train_matrix'])
ubcf_pred = ubcf_model.predict(test_users, k=10)
ubcf_metrics = compute_ranking_metrics(y_true, ubcf_pred, k_values=[5, 10])
results.append({'Approach': 'User-Based CF', **ubcf_metrics})

# Item-Based CF
ibcf_model = ItemBasedCF({'k_neighbors': 30})
ibcf_model.train(dataset['train_matrix'])
ibcf_pred = ibcf_model.predict(test_users, k=10)
ibcf_metrics = compute_ranking_metrics(y_true, ibcf_pred, k_values=[5, 10])
results.append({'Approach': 'Item-Based CF', **ibcf_metrics})

# SVD
svd_model = SVDRecommender({'n_factors': 30})
svd_model.train(dataset['train_matrix'])
svd_pred = svd_model.predict(test_users, k=10)
svd_metrics = compute_ranking_metrics(y_true, svd_pred, k_values=[5, 10])
results.append({'Approach': 'SVD', **svd_metrics})

# Display results
df = pd.DataFrame(results)
df[['Approach', 'ndcg@10', 'recall@10', 'precision@10', 'mrr']]

## Philosophy Comparison

In [ ]:
for model in [pop_model, ubcf_model, ibcf_model, svd_model]:
    philosophy = model.get_philosophy()
    print(f"\n{model.name}:")
    print(f"  Mental Model: {philosophy['mental_model']}")
    print(f"  Strengths: {philosophy['strengths']}")
    print(f"  Best For: {philosophy['best_for']}")